In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/02_silver_cleaning/00_common_functions

In [0]:
df = spark.read.table("hive_metastore.bronze.bronze_medidas")
display(df.limit(5))
df.printSchema()

In [0]:
df = df.select(["TAG","ALR_H_LIM"])

In [0]:
display(df)

In [0]:
col_name = "ALR_H_LIM"

df1 = (
    df
    # trim + remove whitespace anywhere (spaces, tabs, etc.)
    .withColumn("_s", F.regexp_replace(F.trim(F.col(col_name)), r"\s+", ""))
    # keep only digits, sign, comma, dot (drops currency symbols, letters, etc.)
    .withColumn("_s", F.regexp_replace(F.col("_s"), r"[^0-9\+\-\,\.]", ""))
)

In [0]:
# last position of '.' and ',' (0 if not present)
rev = F.reverse(F.col("_s"))

dot_from_end = F.instr(rev, ".")
comma_from_end = F.instr(rev, ",")

last_dot = F.when(dot_from_end > 0, F.length(F.col("_s")) - dot_from_end + 1).otherwise(F.lit(0))
last_comma = F.when(comma_from_end > 0, F.length(F.col("_s")) - comma_from_end + 1).otherwise(F.lit(0))

df2 = (
    df1
    .withColumn("_last_dot", last_dot)
    .withColumn("_last_comma", last_comma)
)

In [0]:
# normalize to a dot-decimal string
normalized = (
    F.when((F.col("_s") == "") | F.col("_s").isNull(), F.lit(None))

    # both exist and dot is the last separator -> dot is decimal, commas are thousands
    .when((F.col("_last_dot") > 0) & (F.col("_last_comma") > 0) & (F.col("_last_dot") > F.col("_last_comma")),
          F.regexp_replace(F.col("_s"), ",", ""))

    # both exist and comma is the last separator -> comma is decimal, dots are thousands
    .when((F.col("_last_dot") > 0) & (F.col("_last_comma") > 0) & (F.col("_last_comma") > F.col("_last_dot")),
          F.regexp_replace(F.regexp_replace(F.col("_s"), r"\.", ""), ",", "."))

    # only comma exists -> comma is decimal
    .when((F.col("_last_comma") > 0) & (F.col("_last_dot") == 0),
          F.regexp_replace(F.col("_s"), ",", "."))

    # only dot exists (or none) -> already ok
    .otherwise(F.col("_s"))
)

df3 = (
    df2
    .withColumn("_norm", normalized)
    .withColumn(f"{col_name}_double", F.expr("try_cast(_norm as double)"))
    .drop("_s", "_last_dot", "_last_comma", "_norm")
)

display(df3)

In [0]:
bad = (
    df3
    .filter(
        F.col(f"{col_name}_double").isNull() &
        F.col(col_name).isNotNull() &
        (F.trim(F.col(col_name)) != "")
    )
    .select(col_name)
    .distinct()
)

display(bad)

In [0]:
df_med = (
    df3
    .withColumn("ALR_H_LIM", F.col("ALR_H_LIM_double"))
    .drop("ALR_H_LIM_double")
)

display(df_med)

In [0]:
df_med.count()

258 378 registos

In [0]:
df_med = df_med.filter(
    (col("TAG").substr(2, 1).isin("P", "S")) &  # Check position 2
    (~col("TAG").substr(7, 1).isin("-", "9", "4"))  # Check position 7
)

In [0]:
# Generate df_arqlmed_U- using filter and like operations
df_med = df_med.filter((col("TAG").rlike("U--$")) | (col("TAG").rlike("0II--$")))


In [0]:
df_med.count()

23692 passou para estes registos

Pivoting the ID column

In [0]:
df_med = df_med.withColumn("TAG_prefix", substring(col("TAG"), 1, 11))

In [0]:
# Group by the first 12 characters and filter groups with more than one distinct ID
temp_df = df_med.groupBy("TAG_prefix").agg(count_distinct("TAG").alias("distinct_count")) \
    .filter(col("distinct_count") > 1)

# Join back with the original DataFrame to filter the relevant rows
df_med = df_med.join(temp_df, "TAG_prefix")

# Show the result
df_med.display()

In [0]:
p_df = df_med.withColumn("TAG_prefix", substring(col("TAG"), 1, 12)) \
           .withColumn("suffix", substring(col("TAG"), -3, 3))

# List of columns to pivot
columns_to_pivot = ["ALR_H_LIM"]


pivoted_dfs = []
for column in columns_to_pivot:
    pivoted_df = p_df.groupBy("TAG_prefix").pivot("suffix").agg(F.first(column))
    
    # Check the column names in the pivoted DataFrame
    print(pivoted_df.columns)
    
    # Rename columns based on expected pivot values
    pivoted_df = pivoted_df.withColumnRenamed("U--", f"{column}_T").withColumnRenamed("I--", f"{column}_I")
    
    # Ensure that renaming reflects actual column names after pivot
    if 'ALR_H_LIM_T' in pivoted_df.columns and 'ALR_H_LIM_I' in pivoted_df.columns:
        pivoted_df = pivoted_df.withColumnRenamed("ALR_H_LIM_T", "H_LIM_V").withColumnRenamed("ALR_H_LIM_I", "H_LIM_C")
    
    pivoted_dfs.append(pivoted_df)

In [0]:
display(pivoted_df)

In [0]:
pivoted_df = pivoted_df.dropna(how='any')

In [0]:
pivoted_df = pivoted_df.withColumn("H_LIM_C", round(col("H_LIM_C"), 2)) \
                       .withColumn("H_LIM_V", round(col("H_LIM_V"), 2))

display(pivoted_df)

In [0]:
dbutils.data.summarize(pivoted_df)

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "silver"
target_table = "silver_medidas"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


In [0]:
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

In [0]:
(
    pivoted_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))


In [0]:
df = spark.read.table("hive_metastore.silver.silver_medidas")
display(df.limit(5))
df.printSchema()

In [0]:
dbutils.data.summarize(df)